# Redis — Introducción práctica

Redis es una base de datos **en memoria**. Guarda datos como pares **clave → valor** y es extremadamente rápida: puede hacer cientos de miles de operaciones por segundo.

No reemplaza a MongoDB ni a PostgreSQL. Vive **al lado** de tu sistema para casos donde la velocidad es crítica:

- Caché de resultados costosos
- Sesiones de usuario con expiración automática
- Contadores en tiempo real
- Rankings y leaderboards
- Colas ligeras de tareas

Lo que veremos hoy:
1. Levantarlo con Docker y conectar desde Python
2. **Strings** — el tipo más básico, con TTL
3. **Hashes** — como un dict, pero en Redis
4. **Lists** — colas e historiales
5. **Sorted Sets** — rankings automáticos

## 0. Levantar Redis con Docker

El `docker-compose.yml` de esta carpeta levanta un Redis en el puerto `6379`:

```yaml
services:
  redis:
    image: redis:7
    ports:
      - "6379:6379"
```

```bash
docker compose up -d
```

Una vez levantado, puedes entrar a la CLI de Redis directamente:

```bash
docker compose exec redis redis-cli
```

Desde ahí puedes ejecutar comandos Redis en crudo. Durante la clase usaremos Python, pero es útil saber que existe.

## Conexión desde Python

```bash
pip install redis
```

In [4]:
import redis

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

r.ping()

True

`decode_responses=True` hace que Redis devuelva strings de Python en lugar de bytes. Sin esa opción, `GET` devolvería `b'valor'` en lugar de `'valor'`.

Antes de empezar, limpiamos la base de datos para partir de cero:

In [5]:
r.flushall()
print("Base de datos limpia")

Base de datos limpia


---

## 1. Strings

El tipo más básico. Una clave apunta a un valor de texto (o número).

```
SET  clave  valor
GET  clave
```

In [6]:
r.set('ciudad', 'Málaga')
r.get('ciudad')

'Málaga'

In [7]:
# Varias claves a la vez
r.mset({'pais': 'España', 'idioma': 'castellano', 'moneda': 'euro'})
r.mget(['pais', 'idioma', 'moneda'])

['España', 'castellano', 'euro']

### Contadores

Si el valor es un número, Redis puede incrementarlo de forma atómica (sin condiciones de carrera aunque haya miles de peticiones simultáneas).

In [8]:
r.set('visitas:home', 0)

r.incr('visitas:home')   # → 1
r.incr('visitas:home')   # → 2
r.incrby('visitas:home', 10)  # → 12

r.get('visitas:home')

'12'

### TTL — expiración automática

Una de las características más útiles de Redis: puedes decirle a una clave cuánto tiempo debe vivir. Cuando expire, Redis la borra solo.

Esto es la base de las **sesiones de usuario** y el **rate limiting**.

In [9]:
import time

# Guardamos un token de sesión que expira en 5 segundos
r.set('sesion:usuario:42', 'token-abc-123', ex=5)

print('TTL justo después de crear:', r.ttl('sesion:usuario:42'), 'segundos')
print('Valor:', r.get('sesion:usuario:42'))

time.sleep(6)

print('\nTTL después de 6 segundos:', r.ttl('sesion:usuario:42'))  # -2 = no existe
print('Valor:', r.get('sesion:usuario:42'))  # None

TTL justo después de crear: 5 segundos
Valor: token-abc-123

TTL después de 6 segundos: -2
Valor: None


`TTL` devuelve:
- Un número positivo → segundos restantes
- `-1` → la clave existe pero no tiene expiración
- `-2` → la clave no existe (o ya expiró)

También puedes añadir o quitar expiración a claves ya existentes:

In [10]:
r.set('config:debug', 'true')
print('TTL inicial:', r.ttl('config:debug'))     # -1, sin expiración

r.expire('config:debug', 60)                     # añadir expiración
print('TTL tras expire:', r.ttl('config:debug')) # ~60

r.persist('config:debug')                        # eliminar expiración
print('TTL tras persist:', r.ttl('config:debug')) # -1 de nuevo

TTL inicial: -1
TTL tras expire: 60
TTL tras persist: -1


---

## 2. Hashes

Un hash en Redis es como un diccionario de Python, pero almacenado bajo una sola clave. Perfecto para representar objetos (usuarios, productos, configuraciones).

```
HSET  clave  campo  valor  campo  valor  ...
HGET  clave  campo
HGETALL clave
```

In [11]:
# Guardamos un perfil de usuario
r.hset('usuario:1', mapping={
    'nombre': 'Ana García',
    'email': 'ana@example.com',
    'plan': 'pro',
    'posts': 0
})

r.hgetall('usuario:1')

{'nombre': 'Ana García',
 'email': 'ana@example.com',
 'plan': 'pro',
 'posts': '0'}

In [12]:
# Leer un campo específico
r.hget('usuario:1', 'email')

'ana@example.com'

In [13]:
# Incrementar un campo numérico
r.hincrby('usuario:1', 'posts', 1)
r.hincrby('usuario:1', 'posts', 1)
r.hincrby('usuario:1', 'posts', 1)

r.hget('usuario:1', 'posts')

'3'

In [14]:
# Actualizar un campo sin tocar el resto
r.hset('usuario:1', 'plan', 'enterprise')

r.hgetall('usuario:1')

{'nombre': 'Ana García',
 'email': 'ana@example.com',
 'plan': 'enterprise',
 'posts': '3'}

In [15]:
# ¿Existe un campo?
print(r.hexists('usuario:1', 'email'))     # True
print(r.hexists('usuario:1', 'telefono')) # False

# ¿Cuántos campos tiene?
print(r.hlen('usuario:1'))

True
False
4


---

## 3. Lists

Una lista ordenada de strings. Puedes añadir elementos por la izquierda (`L`) o por la derecha (`R`) y leerlos en cualquier rango.

Son ideales para **historiales de actividad** y **colas de tareas**.

```
LPUSH  clave  valor   → añade por la izquierda
RPUSH  clave  valor   → añade por la derecha
LRANGE clave  0  -1  → lee todos los elementos
LPOP   clave         → saca el primero
```

In [16]:
# Historial de páginas visitadas por un usuario
# Usamos LPUSH para que la más reciente quede al principio
r.lpush('historial:usuario:1', '/home')
r.lpush('historial:usuario:1', '/productos')
r.lpush('historial:usuario:1', '/carrito')
r.lpush('historial:usuario:1', '/checkout')

# Leer todas las páginas (0 = primero, -1 = último)
r.lrange('historial:usuario:1', 0, -1)

['/checkout', '/carrito', '/productos', '/home']

In [17]:
# Mantener solo las últimas 3 páginas
r.ltrim('historial:usuario:1', 0, 2)
r.lrange('historial:usuario:1', 0, -1)

['/checkout', '/carrito', '/productos']

In [18]:
# Cola de tareas: los trabajos entran por la derecha, se procesan por la izquierda
r.rpush('tareas:pendientes', 'enviar-email:1001')
r.rpush('tareas:pendientes', 'generar-pdf:2034')
r.rpush('tareas:pendientes', 'resize-imagen:img_55')

print('Tareas en cola:', r.llen('tareas:pendientes'))

# Procesar la primera tarea
tarea = r.lpop('tareas:pendientes')
print('Procesando:', tarea)
print('Tareas restantes:', r.llen('tareas:pendientes'))

Tareas en cola: 3
Procesando: enviar-email:1001
Tareas restantes: 2


---

## 4. Sorted Sets

El tipo de dato más potente de Redis. Es un conjunto donde cada elemento tiene una **puntuación** numérica, y el conjunto siempre se mantiene ordenado por esa puntuación automáticamente.

Casos de uso: rankings, leaderboards, feeds cronológicos.

```
ZADD   clave  puntuacion  elemento
ZRANGE clave  0  -1  WITHSCORES   → de menor a mayor
ZREVRANGE clave  0  -1             → de mayor a menor
ZSCORE clave  elemento
ZINCRBY clave  incremento  elemento
ZRANK  clave  elemento              → posición en el ranking
```

In [19]:
# Ranking de un videojuego
r.zadd('ranking:global', {
    'player:ana':    1540,
    'player:carlos': 2310,
    'player:sofia':  1890,
    'player:david':   980,
    'player:lucia':  3100,
})

# Top 3 (de mayor a menor puntuación)
print('Top 3:')
for jugador, puntos in r.zrevrange('ranking:global', 0, 2, withscores=True):
    print(f'  {jugador}: {int(puntos)} pts')

Top 3:
  player:lucia: 3100 pts
  player:carlos: 2310 pts
  player:sofia: 1890 pts


In [20]:
# Posición de un jugador en el ranking (0 = el último, el de menos puntos)
# zrevrank cuenta desde arriba: 0 = el primero
print('Posición de sofia:', r.zrevrank('ranking:global', 'player:sofia') + 1, 'de', r.zcard('ranking:global'))
print('Puntuación de sofia:', r.zscore('ranking:global', 'player:sofia'))

Posición de sofia: 3 de 5
Puntuación de sofia: 1890.0


In [21]:
# Ana gana una partida y suma 200 puntos
nueva_puntuacion = r.zincrby('ranking:global', 200, 'player:ana')
print('Nueva puntuación de ana:', nueva_puntuacion)

# Ranking actualizado completo
print('\nRanking completo:')
for i, (jugador, puntos) in enumerate(r.zrevrange('ranking:global', 0, -1, withscores=True), start=1):
    print(f'  #{i} {jugador}: {int(puntos)} pts')

Nueva puntuación de ana: 1740.0

Ranking completo:
  #1 player:lucia: 3100 pts
  #2 player:carlos: 2310 pts
  #3 player:sofia: 1890 pts
  #4 player:ana: 1740 pts
  #5 player:david: 980 pts


In [22]:
# Filtrar por rango de puntuación: jugadores entre 1500 y 2500 puntos
print('Jugadores entre 1500 y 2500 pts:')
for jugador, puntos in r.zrangebyscore('ranking:global', 1500, 2500, withscores=True):
    print(f'  {jugador}: {int(puntos)} pts')

Jugadores entre 1500 y 2500 pts:
  player:ana: 1740 pts
  player:sofia: 1890 pts
  player:carlos: 2310 pts


---

## 5. Ejercicio final

Implementa un sistema de **rate limiting** básico usando Strings y TTL.

La idea: un usuario solo puede hacer **5 peticiones por minuto** a tu API. Si supera ese límite, se le bloquea hasta que expire el contador.

La función debe:
- Recibir un `user_id`
- Devolver `True` si la petición está permitida
- Devolver `False` si el usuario ha superado el límite
- Usar una clave con TTL de 60 segundos para el contador

In [23]:
LIMITE = 5
VENTANA = 60  # segundos

def permitir_peticion(user_id: str) -> bool:
    clave = f'rate_limit:{user_id}'
    
    contador = r.incr(clave)
    
    if contador == 1:
        # Primera petición de esta ventana: fijamos el TTL
        r.expire(clave, VENTANA)
    
    return contador <= LIMITE


# Simulamos 7 peticiones del mismo usuario
for i in range(1, 8):
    permitido = permitir_peticion('user:99')
    estado = 'OK' if permitido else 'BLOQUEADA'
    contador = r.get('rate_limit:user:99')
    print(f'Petición {i}: {estado} (contador: {contador}, TTL: {r.ttl("rate_limit:user:99")}s)')

Petición 1: OK (contador: 1, TTL: 60s)
Petición 2: OK (contador: 2, TTL: 60s)
Petición 3: OK (contador: 3, TTL: 60s)
Petición 4: OK (contador: 4, TTL: 60s)
Petición 5: OK (contador: 5, TTL: 60s)
Petición 6: BLOQUEADA (contador: 6, TTL: 60s)
Petición 7: BLOQUEADA (contador: 7, TTL: 60s)


---

## Resumen: ¿qué tipo usar?

| Necesito... | Tipo |
|-------------|------|
| Guardar un valor simple o un contador | **String** |
| Representar un objeto con varios campos | **Hash** |
| Mantener un historial o una cola | **List** |
| Elementos únicos sin orden | **Set** |
| Un ranking ordenado automáticamente | **Sorted Set** |
| Que algo desaparezca solo después de X tiempo | **TTL** (en cualquier tipo) |

Redis no es una base de datos principal. Es la capa rápida que **complementa** tu sistema: pones en Redis lo que necesitas acceder en microsegundos, y en MongoDB o PostgreSQL lo que necesitas conservar y consultar con flexibilidad.